In [ ]:
TRAIN_RUN = True

# Installations

In [ ]:
if TRAIN_RUN: 
    ! pip install -q wandb

# Load libraries and setup

In [ ]:
# Import libraries

from zzzs_util import *

import sys
sys.path.append("/kaggle/input/event-detection-ap/")

from metric import *

import os
import gc
import ctypes
import joblib
from tqdm.auto import tqdm

libc = ctypes.CDLL("libc.so.6")

import pytz
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)

import tensorflow as tf
from tensorflow.keras.utils import plot_model
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.layers import Concatenate, LSTM, GRU
from tensorflow.keras.layers import Bidirectional, Multiply
from tensorflow.keras.utils import plot_model

from sklearn.preprocessing import RobustScaler, StandardScaler


COMP_LOC = "/kaggle/input/child-mind-institute-detect-sleep-states/"

In [ ]:
if TRAIN_RUN:
    # Wandb setup

    # Setup user secrets for login
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    WANDB_KEY = user_secrets.get_secret("wandb")

    import wandb
    from wandb.keras import WandbMetricsLogger
    wandb.login(key=WANDB_KEY)
    wandb.init(project='kaggle-zzzs')

# Custom functions

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.metrics import mean_squared_error
from scipy.ndimage import gaussian_filter1d


CHUNK_SIZE = 34560
threshold = 0.6
min_sleep_duration = 30 * 12
activity_tolerance = 30 * 12
order = 6 * 60 * 12

def detect_sleep_events_for_series(predictions, seriesLst, series_lengths):
    all_events = pd.DataFrame(columns=["row_id", "series_id", "step", "event", "score"])
    start_idx = 0
    
    for i, series in tqdm(enumerate(seriesLst), total=len(seriesLst)):
        series_length = series_lengths[series.split(".pkl")[0]]
        num_full_batches = series_length // CHUNK_SIZE
        remainder = series_length % CHUNK_SIZE
        
        # Calculate the total number of batches used for this series
        total_batches = num_full_batches + (1 if remainder > 0 else 0)
        
#         print(series)
#         print(series_length)
#         print(total_batches)
        
        # Extract and concatenate predictions for this series
        series_predictions = []
        for j in range(num_full_batches):
            series_predictions.append(predictions[start_idx + j])
        if remainder > 0:
            series_predictions.append(predictions[start_idx + num_full_batches][:remainder])
        
        series_predictions = np.concatenate(series_predictions)
        start_idx += total_batches
        
#         print(start_idx)
        
        # Apply sigmoid activation
        series_predictions = expit(series_predictions)
        
        # Extract sleep events using predictions
        drop_cols = ['series_id', 'step', 'timestamp']

        test  = pd.read_parquet(f'/kaggle/input/child-mind-institute-detect-sleep-states/train_series.parquet',
                            filters=[('series_id','=',series.split(".pkl")[0])])
        test = test[drop_cols]

        test['timestamp'] = pd.to_datetime(test['timestamp']).apply(lambda t: t.tz_localize(None))

        preds = np.where(series_predictions[:, 0]>threshold, 1, 0)
        probs = series_predictions[:, 0]

        test['prediction'] = preds
        test['prediction'] = test['prediction'].rolling(360+1, center=True).median()
        test['probability'] = probs

        test = test[test['prediction']!=2]

        test.loc[test['prediction']==0, 'probability'] = 1-test.loc[test['prediction']==0, 'probability']
        test['score'] = test['probability'].rolling(60*12*5, center=True, min_periods=10).mean().bfill().ffill()


        test['pred_diff'] = test['prediction'].diff()

        test['event'] = test['pred_diff'].replace({1:'wakeup', -1:'onset', 0:np.nan})

        test_wakeup = test[test['event']=='wakeup'].groupby(test['timestamp'].dt.date).agg('first')
        test_onset = test[test['event']=='onset'].groupby(test['timestamp'].dt.date).agg('last')
        test = pd.concat([test_wakeup, test_onset], ignore_index=True).sort_values('timestamp')
        
        # Convert to the required format
        series_events = pd.DataFrame({
            'row_id': range(len(test)),
            'series_id': [series.split(".pkl")[0]] * len(test),
            'step': test['step'],
            'event': test['event'],
            'score': test['score']
        })
        
        all_events = pd.concat([all_events, series_events], ignore_index=True)
    
    all_events['step'] = all_events['step'].astype(int)
    
    return all_events

In [ ]:
def sorted_directory_listing(directory):
    items = os.listdir(directory)
    sorted_items = sorted(items)
    return sorted_items

# Model Configs

In [ ]:
# Set seed

SEED = 101

np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
tolerances = {
    "onset" : [12, 36, 60, 90, 120, 150, 180, 240, 300, 360],
    'wakeup': [12, 36, 60, 90, 120, 150, 180, 240, 300, 360]    
}

column_names = {
    'series_id_column_name': 'series_id',
    'time_column_name': 'step',
    'event_column_name': 'event',
    'score_column_name': 'score',
}

In [ ]:
# Parameters
NUM_FOLDS = 7 if TRAIN_RUN else 1
NUM_EPOCHS = 1 if TRAIN_RUN else 1
BATCH_SIZE = 2 if TRAIN_RUN else 2
VALID_BATCH_SIZE = 4 if TRAIN_RUN else 4
VALID_EVAL_STEPS = 300 if TRAIN_RUN else 10
# GRADIENT_ACCUMULATION_STEPS = 2

WARMUP_EPOCHS = 1
WARMUP_LR = 1e-5

WEIGHT_DECAY = 1e-4
INIT_LEARNING_RATE = 1e-4
FINAL_LEARNING_RATE = 1e-6

PATIENCE = 3  # For ReduceLROnPlateau
FACTOR = 0.8  # Reduce learning rate by this factor

# LSTM_SEQ_LEN = 128
NUM_FEATS = 121

# Define the number of hours in chunk
NUM_HOURS_CHUNK = 48
CHUNK_SIZE = int(NUM_HOURS_CHUNK * 60 * 60 / 5)

# Modeling parameter functions

In [ ]:
# Custom learning rate scheduler with warmup
def lr_schedule(epoch, lr, warmup_epochs=5, warmup_lr=1e-5):
    if epoch < warmup_epochs:
        return (lr - warmup_lr) / warmup_epochs * epoch + warmup_lr
    else:
        return lr

In [ ]:
def compute_new_lr(epoch, initial_lr=1e-4, final_lr=1e-6, total_epochs=10):
    return initial_lr - (initial_lr - final_lr) * (epoch / total_epochs)

In [ ]:
class CustomLearningRateScheduler(tf.keras.callbacks.Callback):
    def __init__(self, initial_lr, final_lr, total_epochs):
        super(CustomLearningRateScheduler, self).__init__()
        self.initial_lr = initial_lr
        self.final_lr = final_lr
        self.total_epochs = total_epochs

    def on_epoch_end(self, epoch, logs=None):
        new_lr = compute_new_lr(epoch, 
                                initial_lr=self.initial_lr, 
                                final_lr=self.final_lr, 
                                total_epochs=self.total_epochs)
        tf.keras.backend.set_value(self.model.optimizer.lr, new_lr)
        print(f"\nEpoch {epoch+1}: Learning rate is {new_lr:.4f}.")

lr_callback = CustomLearningRateScheduler(INIT_LEARNING_RATE, FINAL_LEARNING_RATE, NUM_EPOCHS)

In [ ]:
# from keras.callbacks import Callback

# class CompetitionMetricCallback(Callback):
#     def __init__(self, val_data, tolerances, series_id_col, time_col, event_col, score_col):
#         super().__init__()
#         self.val_data = val_data
#         self.tolerances = tolerances
#         self.series_id_col = series_id_col
#         self.time_col = time_col
#         self.event_col = event_col
#         self.score_col = score_col
        
#     def on_epoch_end(self, epoch, logs=None):
#         # Get the ground truth from validation data
#         y_true = self.val_data[0]
#         # Use the model to predict the validation data
#         y_pred = self.model.predict(self.val_data[0])
        
#         # Convert y_true and y_pred into the required DataFrame format for scoring
#         solution = ...  # Convert y_true into the required solution format
#         submission = ...  # Convert y_pred into the required submission format
        
#         # Compute the competition metric
#         metric_value = score(solution, submission, self.tolerances, self.series_id_col, self.time_col, self.event_col, self.score_col)
        
#         # Log the metric value
#         logs[f'val_competition_metric'] = metric_value

        
# competition_metric_callback = CompetitionMetricCallback(
#                                                             val_data=val_dataset,
#                                                             tolerances=YOUR_TOLERANCES_DICT,
#                                                             series_id_col=YOUR_SERIES_ID_COLUMN_NAME,
#                                                             time_col=YOUR_TIME_COLUMN_NAME,
#                                                             event_col=YOUR_EVENT_COLUMN_NAME,
#                                                             score_col=YOUR_SCORE_COLUMN_NAME
#                                                         )

# https://www.kaggle.com/code/carlmcbrideellis/zzzs-playing-with-the-competition-metric

In [ ]:
class ValidateEveryXSteps(tf.keras.callbacks.Callback):
    def __init__(self, validation_data, freq):
        super(ValidateEveryXSteps, self).__init__()
        self.validation_data = validation_data
        self.freq = freq
#         self.steps = steps
        self.step_counter = 0

    def on_train_batch_end(self, batch, logs=None):
        self.step_counter += 1
        if self.step_counter % self.freq == 0:

            # if TRAIN_RUN:
            #     val_logs = self.model.evaluate(self.validation_data, verbose=1)
            # else:
            #     val_logs = self.model.evaluate(self.validation_data, steps=validation_steps, verbose=1) 
            
            # # Logging the validation loss
            # if TRAIN_RUN: wandb.log({f"val_loss_every_{self.freq}_steps": val_logs})
            
            # Logging the training loss
            train_loss = logs.get("loss")
            if train_loss:
                if TRAIN_RUN: wandb.log({f"train_loss_every_{self.freq}_steps": train_loss})
            
            print(f"Calculating competition metric for Fold {fold}...")

            
            # You need to iterate over the dataset to collect predictions and labels
            y_true = []
            y_pred = []

            for x_batch, y_batch in tqdm(self.validation_data):
                # Generate predictions for this batch
                batch_pred = model.predict(x_batch, verbose=0)
                # Store the true labels and predictions
                y_true.append(y_batch.numpy())
                y_pred.append(batch_pred)

            # Concatenate all batches
            y_true = np.concatenate(y_true, axis=0)
            predictions = np.concatenate(y_pred, axis=0)

            # if TRAIN_RUN:
            #     predictions = self.model.predict(self.validation_data, verbose=1)
            # else:
            #     predictions = self.model.predict(self.validation_data, steps=validation_steps, verbose=1)
            
            # Calculate loss metric - Mean Squared Error for this example
            mse = mean_squared_error(y_true.flatten(), predictions.flatten())
            # Logging the validation loss
            if TRAIN_RUN: wandb.log({f"val_loss_every_{self.freq}_steps": mse})
    
    
            seriesLst = sorted_directory_listing(f"/kaggle/input/zzzs-numpyfolds-fold-{fold}/fold{fold}/")
            series_lengths = joblib.load(f"/kaggle/input/zzzs-valserieslens/val_serieslens_fold{fold}.pkl")
            val_events = detect_sleep_events_for_series(predictions, seriesLst, series_lengths)

            solution = pd.read_csv("/kaggle/input/zzzs-groupkfold-7folds/train_folds.csv")
            solution = solution[solution['fold']==fold]

            comp_score = score(solution, val_events, tolerances, **column_names)
            
            if TRAIN_RUN: wandb.log({f"train_comp_map_every_{self.freq}_steps": comp_score})

            print(f"Competition metric = {comp_score}")

# Modeling data generator functions

In [ ]:
def split_and_pad_into_chunks(X, y, chunk_size, train=True):
    # Calculate the number of chunks we can form and the size of the last chunk
    num_full_chunks = len(X) // chunk_size
    last_chunk_size = len(X) % chunk_size
    
    # Split the data into full chunks and the last smaller chunk
    X_full_chunks = X[:num_full_chunks * chunk_size].reshape(num_full_chunks, chunk_size, -1)
    y_full_chunks = y[:num_full_chunks * chunk_size].reshape(num_full_chunks, chunk_size, -1)
    
    X_last_chunk = X[num_full_chunks * chunk_size:]
    y_last_chunk = y[num_full_chunks * chunk_size:]
    
    # Pad the last chunk if it's smaller than the required chunk size
    if last_chunk_size > 0:
        padding_size = chunk_size - last_chunk_size
        X_last_chunk = np.pad(X_last_chunk, ((0, padding_size), (0, 0)), mode='constant')
        y_last_chunk = np.pad(y_last_chunk, ((0, padding_size), (0, 0)), mode='constant')
        
        # Add the padded last chunk to our list of chunks
        X_full_chunks = np.vstack((X_full_chunks, X_last_chunk.reshape(1, chunk_size, -1)))
        y_full_chunks = np.vstack((y_full_chunks, y_last_chunk.reshape(1, chunk_size, -1)))
    
    # Filter out chunks without any sleep periods
#     if train:
#         valid_indices = [i for i, y_chunk in enumerate(y_full_chunks) if np.sum(y_chunk) != len(y_chunk)]
#         X_chunks_filtered = X_full_chunks[valid_indices]
#         y_chunks_filtered = y_full_chunks[valid_indices]

    return X_full_chunks, y_full_chunks

In [ ]:
def load_and_preprocess(file_path, chunk_size, train=True):
    
    # Load the specific series data
    X, y, series_id = joblib.load(file_path)
    
    # Split and pad the data into chunks
    X, y = split_and_pad_into_chunks(X, y, chunk_size, train)
    
    return X, y

In [ ]:
def data_generator(fold, chunk_size):
    for train_fold in range(1, 8):
        if train_fold == fold:  # Skip the current validation fold
            continue
            
        train_fold_path = f"/kaggle/input/zzzs-numpyfolds-fold-{train_fold}/fold{train_fold}/"
        for file in os.listdir(train_fold_path):
            X_chunked, y_chunked = load_and_preprocess(train_fold_path + file, chunk_size)
            for i in range(X_chunked.shape[0]):
                yield X_chunked[i], y_chunked[i]


In [ ]:
def validation_data_generator(fold, chunk_size):
    
    validation_fold_path = f"/kaggle/input/zzzs-numpyfolds-fold-{fold}/fold{fold}/"
    
    for file in sorted_directory_listing(validation_fold_path):
        X_chunked, y_chunked = load_and_preprocess(validation_fold_path + file, chunk_size, train=False)
        for i in range(X_chunked.shape[0]):
            yield X_chunked[i], y_chunked[i]


# Model architecture

In [ ]:
# def dnn_model_small():
    
# #     x_input = Input(shape=(train.shape[-2:]))
    
#     x_input = Input(shape=(CHUNK_SIZE, 121))
    
# #     layers = [1024, 512, 384, 256, 128]
# #     layers = [500, 400, 300, 200, 100]
    
#     layers = [256, 128, 64]
    
#     x1 = Bidirectional(LSTM(units=layers[0], return_sequences=True))(x_input)
#     x2 = Bidirectional(LSTM(units=layers[1], return_sequences=True))(x1)
#     x3 = Bidirectional(LSTM(units=layers[2], return_sequences=True))(x2)
# #     x4 = Bidirectional(LSTM(units=layers[3], return_sequences=True))(x3)
# #     x5 = Bidirectional(LSTM(units=layers[4], return_sequences=True))(x4)
    
# #     z2 = Bidirectional(GRU(units=layers[2], return_sequences=True))(x2)
    
# #     z31 = Multiply()([x3, z2])
# #     z31 = BatchNormalization()(z31)
# #     z3 = Bidirectional(GRU(units=layers[3], return_sequences=True))(z31)
    
# #     z41 = Multiply()([x4, z3])
# #     z41 = BatchNormalization()(z41)
# #     z4 = Bidirectional(GRU(units=layers[4], return_sequences=True))(z41)
    
# #     z51 = Multiply()([x5, z4])
# #     z51 = BatchNormalization()(z51)
# #     z5 = Bidirectional(GRU(units=64, return_sequences=True))(z51)
    
# #     x = Concatenate(axis=2)([x5, z2, z3, z4, z5])
    
#     x = Dense(units=128, activation='selu')(x3)
    
#     x_output = Dense(units=1)(x)

#     model = Model(inputs=x_input, outputs=x_output, 
#                   name='DNN_Model')
#     return model

In [ ]:
def dnn_model():
    
#     x_input = Input(shape=(train.shape[-2:]))
    
    x_input = Input(shape=(CHUNK_SIZE, 121))
    
#     layers = [1024, 512, 384, 256, 128]
    layers = [500, 400, 300, 200, 100]
    
#     layers = [256, 128, 64]
    
    x1 = Bidirectional(LSTM(units=layers[0], return_sequences=True))(x_input)
    x2 = Bidirectional(LSTM(units=layers[1], return_sequences=True))(x1)
    x3 = Bidirectional(LSTM(units=layers[2], return_sequences=True))(x2)
    x4 = Bidirectional(LSTM(units=layers[3], return_sequences=True))(x3)
    x5 = Bidirectional(LSTM(units=layers[4], return_sequences=True))(x4)
    
    z2 = Bidirectional(GRU(units=layers[2], return_sequences=True))(x2)
    
    z31 = Multiply()([x3, z2])
    z31 = BatchNormalization()(z31)
    z3 = Bidirectional(GRU(units=layers[3], return_sequences=True))(z31)
    
    z41 = Multiply()([x4, z3])
    z41 = BatchNormalization()(z41)
    z4 = Bidirectional(GRU(units=layers[4], return_sequences=True))(z41)
    
    z51 = Multiply()([x5, z4])
    z51 = BatchNormalization()(z51)
    z5 = Bidirectional(GRU(units=64, return_sequences=True))(z51)
    
    x = Concatenate(axis=2)([x5, z2, z3, z4, z5])
    
    x = Dense(units=128, activation='selu')(x)
    
    x_output = Dense(units=1)(x)

    model = Model(inputs=x_input, outputs=x_output, 
                  name='DNN_Model')
    return model

In [ ]:
if TRAIN_RUN:
    model = dnn_model()
    model.summary()

In [ ]:
if TRAIN_RUN:
    
    plot_model(
                model, 
                to_file='LSTM.png', 
                show_shapes=True,
                show_layer_names=True
            )

In [ ]:
if TRAIN_RUN:
    del model
    free_memory()

# Model training

In [ ]:
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

free_memory()

In [ ]:
# Loop over folds
for fold in range(1, 8):
    
    
    print(f"Fold{fold}")
    
    
    #########################
    # Model Initialization
    #########################
    
    print("Model Initialization")
    
    # Build the model
    model = dnn_model()

    # Compile the model with custom learning rate
    optimizer = tf.keras.optimizers.Adam(learning_rate=INIT_LEARNING_RATE)
    model.compile(optimizer=optimizer, loss='mse')
    
    free_memory()
    
    
    ####################
    # Model Datasets
    ####################
    
    print("Model Datasets Definition")
    
    # Define training and validation datasets
    train_dataset = tf.data.Dataset.from_generator(
        lambda: data_generator(fold, CHUNK_SIZE),
        output_signature=(
            tf.TensorSpec(shape=(None, 121), dtype=tf.float32),
            tf.TensorSpec(shape=(None, 1), dtype=tf.float32)
        )).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)  # Add prefetch for improved performance

    val_dataset = tf.data.Dataset.from_generator(
        lambda: validation_data_generator(fold, CHUNK_SIZE),
        output_signature=(
            tf.TensorSpec(shape=(None, 121), dtype=tf.float32),
            tf.TensorSpec(shape=(None, 1), dtype=tf.float32)
        )).batch(VALID_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
    free_memory()
    
    
    #########################
    # Model Training
    #########################
    
    print("Model training")

    # Steps per epoch
#         steps_per_epoch, validation_steps = joblib.load(f"/kaggle/input/zzzs-calcsteps/steps_fold{fold}.pkl")

#     steps_per_epoch = 350
#     validation_steps = 40

    validation_callback = ValidateEveryXSteps(validation_data=val_dataset, 
#                                               steps=validation_steps, 
                                              freq=VALID_EVAL_STEPS)

    free_memory()
    
    # Train the model
    if TRAIN_RUN:
        callbacks = [lr_callback, validation_callback, WandbMetricsLogger()]
        
        history = model.fit(train_dataset,
                            validation_data=val_dataset,
                            epochs=NUM_EPOCHS,
                            verbose=1,
                            callbacks=callbacks,
                            ) 
 
    else: 
        callbacks = [lr_callback, validation_callback]

        steps_per_epoch = 50
        validation_steps = 10

        history = model.fit(train_dataset,
                        validation_data=val_dataset,
                        epochs=NUM_EPOCHS,
                        steps_per_epoch=steps_per_epoch,
                        validation_steps=validation_steps,
                        verbose=1,
                        callbacks=callbacks,
                        )
    
    
    # After training
    print(f"Predicting on validation set for Fold {fold}...")
    
    val_predictions = model.predict(val_dataset, verbose=1)
    
    # Save the predictions
    save_path = f"./val_predictions_fold_{fold}.npy"
    np.save(save_path, val_predictions)
    
    print(f"Saved validation predictions for Fold {fold} to {save_path}")
    
    print(f"Calculating competition metric for Fold {fold}...")
    
    seriesLst = sorted_directory_listing(f"/kaggle/input/zzzs-numpyfolds-fold-{fold}/fold{fold}/")
    series_lengths = joblib.load(f"/kaggle/input/zzzs-valserieslens/val_serieslens_fold{fold}.pkl")
    val_events = detect_sleep_events_for_series(predictions, seriesLst, series_lengths)
    
    solution = pd.read_csv("/kaggle/input/zzzs-groupkfold-7folds/train_folds.csv")
    solution = solution[solution['fold']==fold]
    
    comp_score = score(solution, val_events, tolerances, **column_names)
    
    print(f"Competition metric = {comp_score}")

    # Log metrics
#     wandb.log({f"train_loss_fold_{fold}": history.history['loss'][-1]})
#     wandb.log({f"val_loss_fold_{fold}": history.history['val_loss'][-1]})

    # Save the trained model
    model.save(f"./model_fold_{fold}.h5")
    
    break


# Archive

In [ ]:
# # Validate train/valid sizes

# # Loop over folds
# for fold in range(1, 8):
    
#     print(f"Fold{fold}")
    
#     #########################
#     # Collect series ids
#     #########################
    
#     print("Collect series ids")
    
#     validation_fold_path = f"/kaggle/input/zzzs-numpyfolds-fold-{fold}/fold{fold}/"
#     validation_series_ids = [file.split('.pkl')[0] for file in os.listdir(validation_fold_path)]
    
#     training_series_ids = []
#     for train_fold in range(1, 8):
#         if train_fold != fold:
#             train_fold_path = f"/kaggle/input/zzzs-numpyfolds-fold-{train_fold}/fold{train_fold}/"
#             training_series_ids.extend([file.split('.pkl')[0] for file in os.listdir(train_fold_path)])
    
#     print(f"Num series train: {len(training_series_ids)}")
#     print(f"Num series valid: {len(validation_series_ids)}")

In [ ]:


# # Loop over folds
# for fold in range(1, 8):
    
#     print(f"Fold{fold}")
    
#     #########################
#     # Model Initialization
#     #########################
    
#     print("Model Initialization")
    
#     # Build the model
#     model = dnn_model()

#     # Compile the model with custom learning rate
#     optimizer = tf.keras.optimizers.Adam(learning_rate=INIT_LEARNING_RATE)
#     model.compile(optimizer=optimizer, loss='mse')
    
#     scaler = joblib.load(f"/kaggle/input/zzzs-stdscalerfit/scaler_fold{fold}.pkl")
    
#     free_memory()
    
        
    
#     #########################
#     # Model Training
#     #########################
    
#     print("Model Training")
    
#     for epoch in range(NUM_EPOCHS):
        
#         print(f"Epoch {epoch+1}")
        
#         # Compute the new learning rate
#         new_lr = compute_new_lr(epoch, 
#                                 initial_lr=INIT_LEARNING_RATE, 
#                                 final_lr=FINAL_LEARNING_RATE, 
#                                 total_epochs=NUM_EPOCHS)
#         tf.keras.backend.set_value(model.optimizer.lr, new_lr)
        
#         # Log learning rate
#         # wandb.log({f"lr_{fold}": new_lr})
    
#         # Training phase: Train on series from all other folds
#         train_losses = []
#         for train_fold in tqdm(range(1, 8)):
#             if train_fold == fold:  # Skip the current validation fold
#                 continue
            
#             train_fold_path = f"/kaggle/input/zzzs-numpyfolds-fold-{train_fold}/fold{train_fold}/"
#             train_fold_losses = []
#             for file in tqdm(os.listdir(train_fold_path)):

#                 # Load the specific series data
#                 with open(train_fold_path + file, 'rb') as f:

#                     X, y, series_id = joblib.load(f)
#                     free_memory()
                    
#                     print("Pickle file loaded")
                    
#                     # Split the data into chunks and filter
#                     X, y = split_and_pad_into_chunks(X, y, CHUNK_SIZE)
#                     free_memory()
                    
#                     print("Chunks generated")
                                
#                     print(f"Series {file.split('.pkl')[0]} training")
                    
#                     # Train the model
#                     history = model.fit(X, y, epochs=1, batch_size=BATCH_SIZE, verbose=1)
#                     free_memory()
                    
#                     print(f"Series {file.split('.pkl')[0]} training completed")

#                     # Log train loss
#                     # wandb.log({f"train_loss_series": history.history['loss'][-1]})
                    
#                     train_fold_losses.append(history.history['loss'][-1])

#                     # Free up memory
#                     del X, y, targets
#                     tf.keras.backend.clear_session()
#                     free_memory()
                
#             train_losses.append(np.mean(train_fold_losses))
        
#         # Log the average validation loss for this fold to wandb
#         # wandb.log({f"train_loss_fold_{fold}": np.mean(val_losses)})

#         # Validation phase: Validate on series from the current fold
#         val_losses = []
#         for file in os.listdir(validation_fold_path):
#             with open(validation_fold_path + file, 'rb') as f:

#                 X, y, series_id = joblib.load(f)

#                 # Evaluate the model
#                 val_loss = model.evaluate(X, y, batch_size=BATCH_SIZE, verbose=0)
#                 val_losses.append(val_loss)

#                 del X, y, data, targets
#                 tf.keras.backend.clear_session()
#                 free_memory()

#         # Log the average validation loss for this fold to wandb
#         # wandb.log({f"val_loss_fold_{fold}": np.mean(val_losses)})
    
#     # Save the trained model
#     model.save(f"./model_fold_{fold}.h5")
    
    
    
    
#     break
    